In [ ]:
import os
import pandas as pd
from tqdm import tqdm
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import roc_auc_score
import numpy as np
import lightgbm as lgb
from sklearn.ensemble import VotingClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.feature_selection import SelectFromModel
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
import pickle
from sklearn.pipeline import Pipeline

In [ ]:
import xgboost  as xgb
from catboost import CatBoostClassifier

# Data Understanding #

In [ ]:
df_start = pd.read_parquet('train_data/train_data_0.pq')

In [ ]:
df_start['id'].nunique()

In [ ]:
target_start = pd.read_csv('train_target.csv')

In [ ]:
target_start['flag'].value_counts()

In [ ]:
df_start_merge = df_start.merge(target_start, on = 'id', how = 'inner')

In [ ]:
df_start_merge.head()

In [ ]:
column = df_start_merge.columns

In [ ]:
print(df_start_merge[column].corr()['flag'].abs().sort_values(ascending = False).to_string())

In [ ]:
df_start_merge.columns

In [ ]:
bin_col = ['pre_since_opened', 'pre_since_confirmed', 'pre_pterm',
       'pre_fterm', 'pre_till_pclose', 'pre_till_fclose',
       'pre_loans_credit_limit', 'pre_loans_next_pay_summ',
       'pre_loans_outstanding',
       'pre_loans_max_overdue_sum', 'pre_loans_credit_cost_rate', 'pre_util', 'pre_over2limit',
       'pre_maxover2limit', 'is_zero_util', 'is_zero_over2limit',
       'is_zero_maxover2limit']

In [ ]:
for col in bin_col:
    print(f'гистограмма для {col}')
    plt.hist(df_start_merge[col], bins = 30)
    plt.show()

In [ ]:
path = 'train_data/'


In [ ]:
#def read_parquet_dataset_from_local(path_to_dataset: str, start_from: int = 0,
                                     num_parts_to_read: int = 2, columns=None, verbose=False) -> pd.DataFrame:
    res = []
    dataset_paths = sorted([os.path.join(path_to_dataset, filename) for filename in os.listdir(path_to_dataset)
                              if filename.startswith('train')])
    print(dataset_paths)

    start_from = max(0, start_from)
    chunks = dataset_paths[start_from: start_from + num_parts_to_read]
    if verbose:
        print('Reading chunks:\n')
        for chunk in chunks:
            print(chunk)
    for chunk_path in tqdm(chunks, desc="Reading dataset with pandas"):
        print('chunk_path', chunk_path)
        chunk = pd.read_parquet(chunk_path,columns=columns)
        res.append(chunk)

    return pd.concat(res).reset_index(drop=True)

In [ ]:
#def prepare_transactions_dataset(path_to_dataset: str, num_parts_to_preprocess_at_once: int = 1, num_parts_total: int=50,
                                 save_to_path=None, verbose: bool=False):
    preprocessed_frames = []

    for step in tqdm(range(0, num_parts_total, num_parts_to_preprocess_at_once),
                                   desc="Transforming transactions data"):
        transactions_frame = read_parquet_dataset_from_local(path_to_dataset, step, num_parts_to_preprocess_at_once,
                                                             verbose=verbose)
        ohe = OneHotEncoder(sparse_output = False, handle_unknown='ignore')


        df_copy = transactions_frame.copy()
        float_cols = df_copy.select_dtypes(include = ['float64']).columns
        int_cols = df_copy.select_dtypes(include = ['int64']).columns
        df_copy[float_cols] = df_copy[float_cols].astype('float32')
        df_copy[int_cols] = df_copy[int_cols].astype('int32')

        ft = ohe.fit_transform(df_copy[['enc_loans_account_holder_type','enc_loans_credit_status','enc_loans_credit_type','enc_loans_account_cur']])
        df_ohe = pd.DataFrame(ft, columns = ohe.get_feature_names_out())
        df_1 = pd.concat([df_copy, df_ohe], axis = 1)


        payment_cols = ['enc_paym_0', 'enc_paym_1', 'enc_paym_2',
       'enc_paym_3', 'enc_paym_4', 'enc_paym_5', 'enc_paym_6', 'enc_paym_7',
       'enc_paym_8', 'enc_paym_9', 'enc_paym_10', 'enc_paym_11', 'enc_paym_12',
       'enc_paym_13', 'enc_paym_14', 'enc_paym_15', 'enc_paym_16',
       'enc_paym_17', 'enc_paym_18', 'enc_paym_19', 'enc_paym_20',
       'enc_paym_21', 'enc_paym_22', 'enc_paym_23', 'enc_paym_24']

        zero_loans_col = [ 'is_zero_loans5', 'is_zero_loans530', 'is_zero_loans3060',
       'is_zero_loans6090', 'is_zero_loans90']
        pre_loans_col  = ['pre_loans5', 'pre_loans530', 'pre_loans3060', 'pre_loans6090', 'pre_loans90']
        #добавляю вес относительно rn
        df_1['weight'] = df_1.groupby('id')['rn'].transform(lambda x: np.exp(-(x.max() - x)) / 25)


        ##разбиение платежей по полугодиям:
        q1_cols = ['enc_paym_0', 'enc_paym_1', 'enc_paym_2',
       'enc_paym_3', 'enc_paym_4', 'enc_paym_5']
        df_1['q1_mean'] = df_1[q1_cols].mean(axis = 1)
        q2_cols =  ['enc_paym_6', 'enc_paym_7',
       'enc_paym_8', 'enc_paym_9', 'enc_paym_10', 'enc_paym_11']
        df_1['q2_mean'] = df_1[q2_cols].mean(axis = 1)
        q3_cols = ['enc_paym_12', 'enc_paym_13',
       'enc_paym_14', 'enc_paym_15', 'enc_paym_16', 'enc_paym_17']
        df_1['q3_mean'] = df_1[q3_cols].mean(axis = 1)
        q4_cols =  ['enc_paym_18', 'enc_paym_19',
       'enc_paym_20', 'enc_paym_21', 'enc_paym_22', 'enc_paym_23', 'enc_paym_24']
        df_1['q4_mean'] = df_1[q4_cols].mean(axis = 1)


        ## пик и минимум платежей
        df_1['enc_min'] = df_1[payment_cols].min(axis = 1)
        df_1['enc_max'] = df_1[payment_cols].max(axis = 1)
        df_1['enc_std'] = df_1[payment_cols].std(axis = 1)

        ## "Временные" признаки
        df_1['minus_pterm_fterm'] = df_1['pre_fterm'] - df_1['pre_pterm']
        df_1['minus_pclose_fclose'] = df_1['pre_till_fclose'] - df_1['pre_till_pclose']
        df_1['since_minus'] = df_1['pre_since_opened'] - df_1['pre_since_confirmed']


        ##Динамика по полугодиям

        df_1['q1_q2_dynamics'] = df_1['q1_mean'] - df_1['q2_mean']
        df_1['q2_q3_dynamics'] = df_1['q2_mean'] - df_1['q3_mean']
        df_1['q3_q4_dynamics'] = df_1['q3_mean'] - df_1['q4_mean']

        ##фин.переменные
        df_1['num_credit'] = df_1.groupby('id')['rn'].transform('count') # количество кредитов
        df_1['count_loans'] = df_1['pre_loans5'] + df_1['pre_loans530'] + df_1['pre_loans3060'] + df_1['pre_loans6090'] + df_1['pre_loans90']
        df_1['overdue_per_credit'] = df_1['count_loans'] / (df_1['num_credit'] + 1)
        df_1['dont_loans_sum'] = df_1[zero_loans_col].sum(axis = 1) #общ. колво дней без просроч
        df_1['loans_sum'] = df_1[pre_loans_col].sum(axis = 1) #общ. колво дней с просроч





        q_cols = ['q1_mean', 'q2_mean', 'q3_mean', 'q4_mean']

        loans_col = [ 'is_zero_loans5', 'is_zero_loans530', 'is_zero_loans3060',
       'is_zero_loans6090', 'is_zero_loans90', 'pre_loans5', 'pre_loans530', 'pre_loans3060', 'pre_loans6090', 'pre_loans90']

        bin_col = ['pre_since_opened', 'pre_since_confirmed', 'pre_pterm',
       'pre_fterm', 'pre_till_pclose', 'pre_till_fclose',
       'pre_loans_credit_limit', 'pre_loans_next_pay_summ',
       'pre_loans_outstanding',
       'pre_loans_max_overdue_sum', 'pre_loans_credit_cost_rate', 'pre_util', 'pre_over2limit',
       'pre_maxover2limit', 'is_zero_util', 'is_zero_over2limit',
       'is_zero_maxover2limit']
        ohe_col = ['enc_loans_account_holder_type_1',
                         'enc_loans_account_holder_type_2',
                        'enc_loans_account_holder_type_3',
                         'enc_loans_account_holder_type_4',
                        'enc_loans_account_holder_type_5',
                         'enc_loans_account_holder_type_6',
                        'enc_loans_credit_status_1',
                        'enc_loans_credit_status_2',
                        'enc_loans_credit_status_3',
                         'enc_loans_credit_status_4',
                        'enc_loans_credit_status_5',
                         'enc_loans_credit_status_6',
                          'enc_loans_credit_type_1',
                          'enc_loans_credit_type_2',
                        'enc_loans_credit_type_3',
                          'enc_loans_credit_type_4',
                        'enc_loans_credit_type_5',
                        'enc_loans_account_cur_1',
                        'enc_loans_account_cur_2',
                        'enc_loans_account_cur_3']

        new_col = ['num_credit', 'count_loans', 'overdue_per_credit', 'dont_loans_sum', 'loans_sum']
        trend_col = ['pre_loans_credit_limit', 'pre_loans_next_pay_summ',
       'pre_loans_outstanding',
       'pre_loans_max_overdue_sum', 'pre_loans_credit_cost_rate', 'pre_util', 'pre_over2limit',
       'pre_maxover2limit', 'is_zero_util', 'is_zero_over2limit',
       'is_zero_maxover2limit']


        # макс мин сред в зависимости от типа перемененых
        agg_dict = {}
        for col in bin_col:
                agg_dict[col] = ['mean','max', 'min', 'sum']
        for col in loans_col:
                agg_dict[col] = ['sum']
        for col in payment_cols:
                agg_dict[col] = ['mean','max', 'min', 'sum']
        for col in q_cols:
                agg_dict[col] = ['mean','max', 'min', 'sum']
        for col in ohe_col:
                agg_dict[col] = ['sum']
        for col in new_col:
                agg_dict[col] = ['mean']


        df_1 = df_1.sort_values(['id', 'rn'])
        df_group  = df_1.groupby('id').agg(agg_dict).reset_index()


        df_group .columns = ['_'.join(col).strip() if isinstance(col, tuple) else col
                    for col in df_group .columns.values]








        ##статусы кредита

        statuses_diff = df_1.groupby('id')['enc_loans_credit_status'].diff()
        df_1['_status_worsen'] = (statuses_diff > 0).astype(int)
        df_1['_status_improve'] = (statuses_diff < 0).astype(int)
        df_1['_status_changes'] = (statuses_diff < 0).astype(int)

        status_agg = df_1.groupby('id').agg(
            status_worsen=('_status_worsen', 'sum'),
            status_improve=('_status_improve', 'sum'),
            status_changes=('_status_changes', 'sum'),
            status_first=('enc_loans_credit_status', 'first'),
            status_last=('enc_loans_credit_status', 'last'),
        ).reset_index()
        status_agg['change'] = status_agg['status_last'] - status_agg['status_first']

        first_last = df_1.groupby('id')[trend_col].agg(['first','last'])
        first_last.columns = ['_'.join(col).strip() for col in first_last.columns.values]
        for col in trend_col:
            first_last[f'{col}_diff'] = first_last[f'{col}_first'] - first_last[f'{col}_last']
        first_last = first_last[[f'{col}_diff' for col in trend_col]].reset_index()

        def compute_weighted_trends_vectorized(df, id_col, x_col,w_col,trend_cols):
            df = df.sort_values([id_col,x_col])
            sum_w = df.groupby(id_col)[w_col].transform('sum')
            wx = df[w_col]*df[x_col]
            x_mean = wx.groupby(df[id_col]).transform('sum') / sum_w
            dx = df[x_col] - x_mean
            result = pd.DataFrame(index=df.index)
            for col in trend_cols:
                wy = df[w_col]*df[col]
                y_mean = wy.groupby(df[id_col]).transform('sum')/sum_w
                dy = df[col] - y_mean

                num_sum = (df[w_col] *dx *dy).groupby(df[id_col]).transform('sum')
                den_sum = (df[w_col] * dx **2).groupby(df[id_col]).transform('sum')
                result[f'{col}_trend'] = (num_sum/den_sum.replace(0,np.nan)).fillna(0)

            result[id_col]=df[id_col].values
            return result.groupby(id_col).first().reset_index()
        trend_result = compute_weighted_trends_vectorized(df_1,'id','rn','weight',trend_col)



        for df in [df_group, status_agg, first_last, trend_result]:
            for col in df.columns:
                if col in ['id', 'id_', 'index'] and col != 'id':
                    df.rename(columns={col: 'id'}, inplace=True)
        df_dif_trend  = (
    status_agg
    .merge(first_last, on='id', how='left')
    .merge(trend_result, on='id', how='left'))
        df_group = df_group.merge(df_dif_trend, on='id', how='left')

















        transactions_frame = df_group









        if save_to_path:
            block_as_str = str(step)
            if len(block_as_str) == 1:
                block_as_str = '00' + block_as_str
            else:
                block_as_str = '0' + block_as_str
            transactions_frame.to_parquet(os.path.join(save_to_path, f'processed_chunk_{block_as_str}.parquet'))

        preprocessed_frames.append(transactions_frame)
    return pd.concat(preprocessed_frames)

In [ ]:
#data = prepare_transactions_dataset(path, num_parts_to_preprocess_at_once=2, num_parts_total=12,
                                 save_to_path='train_data/')

Transforming transactions data:   0%|                                                            | 0/6 [00:00<?, ?it/s]

['train_data/train_data_0.pq', 'train_data/train_data_1.pq', 'train_data/train_data_10.pq', 'train_data/train_data_11.pq', 'train_data/train_data_2.pq', 'train_data/train_data_3.pq', 'train_data/train_data_4.pq', 'train_data/train_data_5.pq', 'train_data/train_data_6.pq', 'train_data/train_data_7.pq', 'train_data/train_data_8.pq', 'train_data/train_data_9.pq']



Reading dataset with pandas:   0%|                                                               | 0/2 [00:00<?, ?it/s]

chunk_path train_data/train_data_0.pq



Reading dataset with pandas:  50%|███████████████████████████▌                           | 1/2 [00:00<00:00,  1.60it/s]

...


Reading dataset with pandas:   0%|                                                               | 0/2 [00:00<?, ?it/s]

chunk_path train_data/train_data_8.pq



Reading dataset with pandas:  50%|███████████████████████████▌                           | 1/2 [00:01<00:01,  1.11s/it]

chunk_path train_data/train_data_9.pq



Transforming transactions data: 100%|███████████████████████████████████████████████████| 6/6 [16:11<00:00, 161.91s/it]


In [ ]:
targets = pd.read_csv('train_target.csv')

In [ ]:
df = data.merge(targets, on = 'id', how = 'right')

In [ ]:
print(targets.shape)
print(data.shape)
print(df.shape)

(3000000, 2)
(3000000, 248)
(3000000, 249)


In [ ]:
#df.to_parquet('full_set_4.parquet')

**Загрузка данных и начало обучения**

In [ ]:
df_1 = pd.read_parquet('full_set_4.parquet')

**Feature selection**

In [ ]:
correlations = df_1.corrwith(df_1['flag']).abs().sort_values(ascending=False)
print(correlations.to_string())

flag                                   1.000000
q1_mean_mean                           0.092437
enc_paym_2_mean                        0.086344
enc_paym_1_mean                        0.085928
enc_paym_3_mean                        0.083111
dont_loans_sum_mean                    0.081999
enc_paym_4_mean                        0.078657
enc_paym_5_mean                        0.074095
enc_paym_6_mean                        0.069308
q2_mean_mean                           0.067959
enc_paym_7_mean                        0.066365
enc_paym_0_mean                        0.064780
enc_paym_8_mean                        0.064296
q1_mean_min                            0.063672
enc_paym_9_mean                        0.062394
q2_mean_min                            0.059723
enc_paym_10_mean                       0.059347
enc_paym_11_mean                       0.057489
q3_mean_min                            0.057393
q3_mean_mean                           0.057050
enc_paym_12_mean                       0

*Слабые линейные зависимости. Беру lgb для поиска нелинейных зависимостей и выявления наиболее эффективных переменных*

# Modeling #

**Base_models**

In [ ]:
drop_cols = ['flag','id']

In [ ]:
X = df_1.drop(drop_cols, axis = 1)
y = df_1['flag']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size = 0.2, random_state = 42, stratify = y)

In [ ]:
X_train.shape

(2400000, 247)

In [ ]:
selector = SelectFromModel(
    lgb.LGBMClassifier(
        n_estimators=100,
        learning_rate=0.05,
        num_leaves=31,
        scale_pos_weight=27.18,
        random_state=42,
        verbose=-1
    ),
    max_features=70,
    threshold=-np.inf
)

X_train_selected = selector.fit_transform(X_train, y_train)
X_test_selected = selector.transform(X_test)

print(f"Признаков до: {X_train.shape[1]}")
print(f"Признаков после: {X_train_selected.shape[1]}")

Признаков до: 247
Признаков после: 70


In [ ]:
X_tr, X_val, y_tr, y_val = train_test_split(X_train_selected, y_train, test_size = 0.5, random_state = 42, stratify = y_train)

In [ ]:
X_tr.shape

(1200000, 70)

In [ ]:
lgb_base = lgb.LGBMClassifier(
    n_estimators=400,
    learning_rate=0.02,
    num_leaves=95,
    max_depth=9,
    scale_pos_weight=27.18,
    min_child_samples=30,
    min_split_gain=0.05,
    subsample=0.7,
    colsample_bytree=0.6,
    subsample_freq=10,
    n_jobs=-1,
    random_state=42,
    verbose=-1
)

In [ ]:
xgb_base =  xgb.XGBClassifier(
    n_estimators=500,
    learning_rate=0.03,
    max_depth=5,
    min_child_weight=3,
    gamma=0.05,
    subsample=0.8,
    colsample_bytree=0.7,
    colsample_bylevel=0.7,
    tree_method='hist',
    max_bin=128,
    random_state=42,
    n_jobs=-1,
    use_label_encoder=False,
    verbosity=0,
    eval_metric='auc')

In [ ]:
catboost_base =  CatBoostClassifier(
    iterations = 500,
    thread_count=-1,
    verbose=False,
    random_seed=42,
  eval_metric='AUC')

In [ ]:
catboost_base.fit(X_train_selected, y_train)

In [ ]:
xgb_base.fit(X_train_selected, y_train)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,0.7
,colsample_bynode,None
,colsample_bytree,0.7
,device,None
,early_stopping_rounds,None
,enable_categorical,True
,eval_metric,'auc'


In [ ]:
lgb_base.fit(X_train_selected, y_train)

,boosting_type,'gbdt'
,num_leaves,95
,max_depth,9
,learning_rate,0.02
,n_estimators,400
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.05
,min_child_weight,0.001
,min_child_samples,30


**Предсказания**

In [ ]:
pred_train_xgb = xgb_base.predict_proba(X_train_selected)[:, 1]
pred_train_lgb = lgb_base.predict_proba(X_train_selected)[:, 1]
pred_train_cat = catboost_base.predict_proba(X_train_selected)[:, 1]

In [ ]:
pred_test_start_xgb = xgb_base.predict_proba(X_test_selected)[:,1]
pred_test_start_lgb = lgb_base.predict_proba(X_test_selected)[:,1]
pred_test_start_catb = catboost_base.predict_proba(X_test_selected)[:,1]

In [ ]:
print(f'start_xgb roc auc score tr {roc_auc_score(y_train, pred_train_xgb)}')
print(f'start_lgb roc auc score tr{roc_auc_score(y_train, pred_train_lgb)}')
print(f'start_catb roc auc score tr {roc_auc_score(y_train, pred_train_cat)}')

start_xgb roc auc score tr 0.7556944315860951
start_lgb roc auc score tr0.7723420749966677


100 фичей
start_xgb roc auc score tr 0.7565210875480541
start_lgb roc auc score tr0.7739328329695981
start_catb roc auc score tr 0.7785508272063636

*с остановкой* start_xgb roc auc score tr 0.7681333980738668
start_lgb roc auc score tr0.6829124599286658
start_catb roc auc score tr 0.7792157006223953

In [ ]:
print(f'start_xgb roc auc score {roc_auc_score(y_test, pred_test_start_xgb)}')
print(f'start_lgb roc auc score {roc_auc_score(y_test, pred_test_start_lgb)}')
print(f'start_catb roc auc score {roc_auc_score(y_test, pred_test_start_catb)}')

start_xgb roc auc score 0.7449711773017704
start_lgb roc auc score 0.7469807302650853


100 фичей

start_xgb roc auc score 0.745158242105917

start_lgb roc auc score 0.7465921257391637

start_catb roc auc score 0.7410522099260257

*с остановкой*  

start_xgb roc auc score 0.7488997172122109

start_lgb roc auc score 0.6758299906834407

start_catb roc auc score 0.7466002934011026

**Optuna подбор**

**lgb**

In [ ]:
study = optuna.create_study(sampler= optuna.samplers.TPESampler(seed=42, n_startup_trials=5,
                                                                 n_ei_candidates = 64,multivariate=True,),
                            pruner = MedianPruner(
    n_startup_trials=10,
    n_warmup_steps=20,
    interval_steps=10
),
                            study_name = 'my_first_study',
                            direction = 'maximize')

C:\Users\Тарас\AppData\Local\Temp\ipykernel_9100\405873180.py:1: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  study = optuna.create_study(sampler= optuna.samplers.TPESampler( seed=42, n_startup_trials=5,
[I 2026-08-15 10:26:03,635] A new study created in memory with name: my_first_study


In [ ]:
def objective(trial):
    estimator = trial.suggest_int('n_estimators',400,700)
    learning_rate = trial.suggest_float('learning_rate',0.01,0.05)
    num_leaves = trial.suggest_int('num_leaves', 80,95)
    max_depth = trial.suggest_int('max_depth', 7,13)
    min_child_samples  = trial.suggest_int('min_child_samples', 23, 35)
    min_split_gain = trial.suggest_float('min_split_gain',0.03,0.07)
    subsample = trial.suggest_float('subsample', 0.7,1)
    colsample_bytree = trial.suggest_float('colsample_bytree',0.4,0.8)
    subsample_freq = trial.suggest_int('subsample_freq', 8,14)

    model  = lgb.LGBMClassifier(
    n_estimators=estimator,
    learning_rate=learning_rate,
    num_leaves=num_leaves,
    max_depth=max_depth,
    scale_pos_weight=27.18,
    min_child_samples=min_child_samples,
    min_split_gain=min_split_gain,
    subsample=subsample,
    colsample_bytree=colsample_bytree,
    subsample_freq=subsample_freq,
    n_jobs=-1,
    random_state=42,
    verbose=-1)
    scores = cross_val_score(model , X_train_selected, y_train,cv=3,scoring ='roc_auc', n_jobs = 1)
    return scores.mean()


In [ ]:
study.optimize(objective, n_trials = 50,show_progress_bar=True)

  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-08-15 10:57:34,594] Trial 32 finished with value: 0.7468946774518357 and parameters: {'n_estimators': 551, 'learning_rate': 0.019600686444308728, 'num_leaves': 94, 'max_depth': 7, 'min_child_samples': 34, 'min_split_gain': 0.037715429354750876, 'subsample': 0.8950786810804988, 'colsample_bytree': 0.46109532885002, 'subsample_freq': 9}. Best is trial 32 with value: 0.7468946774518357.
[I 2026-08-15 10:59:28,112] Trial 33 finished with value: 0.7440687999000527 and parameters: {'n_estimators': 521, 'learning_rate': 0.012145463100709963, 'num_leaves': 92, 'max_depth': 7, 'min_child_samples': 34, 'min_split_gain': 0.03591615839309697, 'subsample': 0.9795844424998155, 'colsample_bytree': 0.46562031598112874, 'subsample_freq': 9}. Best is trial 32 with value: 0.7468946774518357.
[I 2026-08-15 11:01:13,973] Trial 34 finished with value: 0.7472793853952086 and parameters: {'n_estimators': 557, 'learning_rate': 0.029693893042730887, 'num_leaves': 93, 'max_depth': 7, 'min_child_samples':

**Best trial: 18. Best value: 0.745425:**

In [ ]:
best_params = study.best_params

In [ ]:
best_params

{'n_estimators': 700,
 'learning_rate': 0.0274810297509568,
 'num_leaves': 91,
 'max_depth': 8,
 'min_child_samples': 35,
 'min_split_gain': 0.055560815471208724,
 'subsample': 0.9771068534341079,
 'colsample_bytree': 0.4305389196160982,
 'subsample_freq': 8}

{'n_estimators': 700,
 'learning_rate': 0.0274810297509568,
 'num_leaves': 91,
 'max_depth': 8,
 'min_child_samples': 35,
 'min_split_gain': 0.055560815471208724,
 'subsample': 0.9771068534341079,
 'colsample_bytree': 0.4305389196160982,
 'subsample_freq': 8}

In [ ]:
model = lgb.LGBMClassifier(
    **best_params)

In [ ]:
model.fit(X_train_selected, y_train)

,boosting_type,'gbdt'
,num_leaves,91
,max_depth,8
,learning_rate,0.0274810297509568
,n_estimators,700
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.055560815471208724
,min_child_weight,0.001
,min_child_samples,35


In [ ]:
pred_test_lgb_best = model.predict_proba(X_test_selected)[:,1]

In [ ]:
print(f'lgb roc auc score {roc_auc_score(y_test, pred_test_lgb_best)}')

lgb roc auc score 0.7495555030487457


**lgb roc auc score 0.7495555030487457**

**подбор xgb**

In [ ]:
study_xgb = optuna.create_study(sampler = optuna.samplers.TPESampler(seed=42, n_startup_trials=5,
                                                                 n_ei_candidates = 64,multivariate=True),
                                pruner = MedianPruner(
    n_startup_trials=10,
    n_warmup_steps=15,
    interval_steps=10
), study_name = 'XGB_first_study',
                            direction = 'maximize')


C:\Users\Тарас\AppData\Local\Temp\ipykernel_9100\1544536056.py:1: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  study_xgb = optuna.create_study(sampler = optuna.samplers.TPESampler(seed=42, n_startup_trials=5,
[I 2026-08-15 13:40:53,810] A new study created in memory with name: XGB_first_study


In [ ]:
def objective(trial):
    estimator = trial.suggest_int('n_estimators',400,700)
    learning_rate = trial.suggest_float('learning_rate',0.01,0.05)
    max_depth = trial.suggest_int('max_depth', 5,11)
    min_child_weight  = trial.suggest_int('min_child_weight', 2, 5)
    gamma = trial.suggest_float('gamma',0.03,0.07)
    subsample = trial.suggest_float('subsample', 0.7,1)
    colsample_bytree = trial.suggest_float('colsample_bytree',0.6,1)
    colsample_bylevel = trial.suggest_float('colsample_bylevel',0.6,1)


    model = xgb.XGBClassifier(
    n_estimators=estimator,
    learning_rate=learning_rate,
    max_depth=max_depth,
    min_child_weight=min_child_weight,
    gamma=gamma,
    subsample=subsample,
    colsample_bytree=colsample_bytree,
    colsample_bylevel=colsample_bylevel,
    tree_method='hist',
    max_bin=128,
    random_state=42,
    n_jobs=-1,
    use_label_encoder=False,
    verbosity=0,
    eval_metric='auc')
    scores = cross_val_score(model, X_train_selected, y_train,cv=3,scoring ='roc_auc', n_jobs = 1)
    return scores.mean()


In [ ]:
study_xgb.optimize(objective, n_trials = 50,show_progress_bar=True)

  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-08-15 13:55:47,740] Trial 4 finished with value: 0.7415033897772968 and parameters: {'n_estimators': 419, 'learning_rate': 0.047955421490133335, 'max_depth': 11, 'min_child_weight': 5, 'gamma': 0.04218455076693483, 'subsample': 0.7293016342019151, 'colsample_bytree': 0.8736932106048627, 'colsample_bylevel': 0.7760609974958406}. Best is trial 4 with value: 0.7415033897772968.
[I 2026-08-15 13:57:08,752] Trial 5 finished with value: 0.7437967131419861 and parameters: {'n_estimators': 436, 'learning_rate': 0.029807076404450808, 'max_depth': 5, 'min_child_weight': 5, 'gamma': 0.04035119926400068, 'subsample': 0.8987566853061946, 'colsample_bytree': 0.7246844304357644, 'colsample_bylevel': 0.8080272084711243}. Best is trial 5 with value: 0.7437967131419861.
[I 2026-08-15 14:00:38,084] Trial 6 finished with value: 0.7475417688268443 and parameters: {'n_estimators': 564, 'learning_rate': 0.017394178221021083, 'max_depth': 11, 'min_child_weight': 5, 'gamma': 0.06757995766256758, 'subsa

In [ ]:
best_params_xgb = study_xgb.best_params

In [ ]:
best_params_xgb

{'n_estimators': 696,
 'learning_rate': 0.024931423486146076,
 'max_depth': 8,
 'min_child_weight': 4,
 'gamma': 0.045697152377670934,
 'subsample': 0.9757811532458035,
 'colsample_bytree': 0.6340574547080189,
 'colsample_bylevel': 0.743437837773719}

In [ ]:
model_xgb = xgb.XGBClassifier(**best_params_xgb, device='cuda:0',  tree_method='hist')

In [ ]:
print(f"Версия XGBoost: {xgb.__version__}")

Версия XGBoost: 3.3.0


In [ ]:
model_xgb.fit(X_train_selected, y_train)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,0.743437837773719
,colsample_bynode,None
,colsample_bytree,0.6340574547080189
,device,'cuda:0'
,early_stopping_rounds,None
,enable_categorical,True
,eval_metric,None


In [ ]:
pred_test_xgb_best = model_xgb.predict_proba(X_test_selected)[:,1]

In [ ]:
print(f'lgb roc auc score {roc_auc_score(y_test, pred_test_xgb_best)}')

lgb roc auc score 0.7502045195212212


**Ансамбль моделей blenjding**

In [ ]:
estimator = [('lgb', lgb.LGBMClassifier(**best_params)),
                      ('xgb',  xgb.XGBClassifier(**best_params_xgb))]

In [ ]:
ensemble = VotingClassifier(estimators = estimator, voting = 'soft')

In [ ]:
ensemble.fit(X_train_selected, y_train)

,estimators,"[('lgb', ...), ('xgb', ...)]"
,voting,'soft'
,weights,None
,n_jobs,None
,flatten_transform,True
,verbose,False
,boosting_type,'gbdt'
,num_leaves,91
,max_depth,8
,learning_rate,0.0274810297509568
,n_estimators,700


In [ ]:
ensemble_pred = ensemble.predict_proba(X_test_selected)[:,1]

In [ ]:
ensemble_pred_train = ensemble.predict_proba(X_train_selected)[:,1]

In [ ]:
print(f"Предсказание train: {roc_auc_score(y_train, ensemble_pred_train)}")

Предсказание train: 0.8029541140871552


In [ ]:
print(f"Предсказание test: {roc_auc_score(y_test, ensemble_pred)}")

Предсказание test: 0.7504306085370553


предыдущее 0.7504306085370553

In [ ]:
with open('ensemble_blending_1.pkl', 'wb') as f:
    pickle.dump(ensemble, f)

**Stacking**

In [ ]:
meta_model = LogisticRegression(
    class_weight='balanced',
    C=0.1,
    max_iter=2000,
    random_state=42,
    solver='liblinear'
)


stacking_lgb = StackingClassifier(
    estimators=[
        ('lgb', lgb.LGBMClassifier(**best_params)),
        ('xgb', xgb.XGBClassifier(**best_params_xgb))
    ],
    final_estimator=meta_model,
    cv=3,
    stack_method='predict_proba',
    n_jobs=-1)

In [ ]:
stacking_lgb.fit(X_train_selected, y_train)

In [ ]:
pred_train_lgb_fin = stacking_lgb.predict_proba(X_train_selected)[:, 1]
pred_test_lgb_fin = stacking_lgb.predict_proba(X_test_selected)[:, 1]

In [ ]:
print(f'stacking_train {roc_auc_score(y_train, pred_train_lgb_fin)}')
print(f'stacking test {roc_auc_score(y_test, pred_test_lgb_fin)}')

**Пайплайн**

In [ ]:
pipeline = Pipeline([('preprocessor',selector),('classifier',ensemble)])

In [ ]:
pipeline.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,estimator,LGBMClassifie... verbose=-1)
,threshold,-inf
,prefit,False
,norm_order,1
,max_features,70
,importance_getter,'auto'
,boosting_type,'gbdt'


In [ ]:
pipeline_pred = pipeline.predict_proba(X_train)[:,1]
pipeline_pred_test = pipeline.predict_proba(X_test)[:,1]
print(f"Предсказание train: {roc_auc_score(y_train, pipeline_pred)}")
print(f"Предсказание test: {roc_auc_score(y_test, pipeline_pred_test)}")

Предсказание train: 0.8029541140871552
Предсказание test: 0.7504306085370553


In [ ]:
with open('pipeline.pkl', 'wb') as f:
    pickle.dump(pipeline, f)

In [ ]:
test_ids = list(range(len(pipeline_pred_test)))

In [ ]:
final_predict = pd.DataFrame({'id': test_ids, 'predict' : pipeline_pred_test})

In [ ]:
final_predict.to_csv('final_predict.csv', index=False)